In [ ]:
import jax
import xarray as xr
import pyqg
import numpy as np

from matplotlib import pyplot as plt

In [ ]:
nx, ny = 256, 256
Lx, Ly = 1e6, 1e6
Ld = 15000.0
hour = 3600  # sec
day = 24 * hour

T = 40 * day
dt = hour / 4
nsteps = int(T / dt)
interval_steps = 24

# initialize PV anomalies to a plane wave
x = np.linspace(0, Lx, nx, endpoint=False)
y = np.linspace(0, Ly, ny, endpoint=False)
xgrid, ygrid = np.meshgrid(x, y, indexing="xy")

k = 10 * (2 * np.pi / Lx)
l = 2 * (2 * np.pi / Ly)
print(f"k * Ld = {k * Ld}")
print(f"l * Ld = {l * Ld}")

gaussian_radius = Ly / 5
gaussian_envelope = np.exp(-((ygrid - Ly / 2) ** 2) / (gaussian_radius**2)) * np.exp(
    -((xgrid - Lx / 2) ** 2) / (gaussian_radius**2)
)
q_upper = -np.cos(k * xgrid + l * ygrid) * 1e-5  # * gaussian_envelope * 1e-5
q_lower = q_upper.copy()

pyqg_model = pyqg.QGModel(
    nx=nx, ny=ny, L=Ly, W=Lx, rd=Ld, delta=1.0, U1=0.0, U2=0.0, rek=0.0, tmax=T
)
pyqg_model.set_q1q2(q_upper, q_lower)

for _ in pyqg_model.run_with_snapshots(tsnapstart=0, tsnapint=10):
    plt.pcolormesh(pyqg_model.q[0, :, :])
    plt.colorbar()
    plt.show()

In [ ]:
plt.pcolormesh(pyqg_model.q[0, :, :])

In [ ]:
ds = xr.open_dataset("../output/examples/barotropic_rossby_wave.zarr", engine="zarr")
ds

In [ ]:
ds["psi"].isel(time=40).sel(lev=0).plot()

In [ ]:
time = 0

q = ds["q"].isel(time=time, lev=0)
u = ds["u"].isel(time=time, lev=0)
v = ds["v"].isel(time=time, lev=0)

plt.pcolormesh(ds["x"] / 1000, ds["y"] / 1000, q, cmap="RdBu_r", vmin=-2e-5, vmax=2e-5)
plt.colorbar()
plt.quiver(ds["x"][::10] / 1000, ds["y"][::10] / 1000, u[::10, ::10], v[::10, ::10])

In [ ]:
def second_derivative(da, dim):
    """
    Estimate the second derivative of a periodic DataArray
    along a given dimension.
    """

    dx = da[dim].diff(dim)[0].values

    return (da.roll({dim: 1}) + da.roll({dim: -1}) - 2 * da) / dx**2


beta = ds.attrs["beta"]
(beta - second_derivative(u, "y")).plot()